## training data 0000


```bash
OUT_DIR: ./tahoe_scgpt_single_target_log1p
[label_vocab.json] keys: ['gene2label', 'label2gene']
  num_classes: 94
  gene2label preview: [('ABCB1', 0), ('ABL1', 1), ('ACE', 2), ('ADRB1', 3), ('ADRB2', 4), ('AGTR1', 5), ('AHR', 6), ('ALDH2', 7), ('ALOX5', 8), ('ANPEP', 9)]

[Parquet files] total: 53
  ood: 4 files
  test: 5 files
  train: 39 files
  val: 5 files

==================== ood ====================

File[0]: ./tahoe_scgpt_single_target_log1p/ood_test_0000.parquet
Schema:
genes: list<element: int64>
  child 0, element: int64
expressions: list<element: double>
  child 0, element: double
label: int64
target_gene: string
drug: string
cell_line_id: string
sample: string
plate: string
split: string

Quick stats on first 2000 rows:
  - n_rows: 2000
  - split_counts: {'ood_test': 2000}
  - label_counts_top10: {40: 278, 88: 170, 92: 113, 20: 109, 1: 104, 75: 103, 0: 93, 25: 92, 55: 90, 24: 82}
  - genes_len: {'min': 409, 'max': 2048, 'mean': 962.759}
  - expr_len: {'min': 409, 'max': 2048, 'mean': 962.759}
  - len_mismatch_rows: 0
  - genes_value_types_top5: {'ndarray': 2000}
  - expr_value_types_top5: {'ndarray': 2000}

--- ood sample from ood_test_0000.parquet: showing 3 rows ---
[Row 0] split=ood_test label=25 target_gene=COMT drug=Tolcapone
        cell_line_id=CVCL_0179 sample=smp_1813 plate=plate4

[Row 1] split=ood_test label=75 target_gene=REN drug=Aliskiren
        cell_line_id=CVCL_0179 sample=smp_1815 plate=plate4

[Row 2] split=ood_test label=55 target_gene=MTOR drug=Temsirolimus
        cell_line_id=CVCL_0028 sample=smp_1802 plate=plate4

```

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

PARQUET_PATH = "./tahoe_scgpt_single_target_log1p/train_0000.parquet"

# 不要一次性 to_pandas() 全文件
table = pq.read_table(PARQUET_PATH)
print("Total rows:", table.num_rows)
print(table.schema)
print()


Total rows: 50000
genes: list<element: int64>
  child 0, element: int64
expressions: list<element: double>
  child 0, element: double
label: int64
target_gene: string
drug: string
cell_line_id: string
sample: string
plate: string
split: string



In [6]:

row_idx = 2
df_one = table.slice(row_idx, 1).to_pandas()
row = df_one.iloc[0]

print("split:", row["split"])
print("label:", row["label"])
print("target_gene:", row["target_gene"])
print("drug:", row["drug"])
print("cell_line_id:", row["cell_line_id"])
print("sample:", row["sample"])
print("plate:", row["plate"])

genes = row["genes"]
exprs = row["expressions"]

print("\nGenes length:", len(genes))
print("Expressions length:", len(exprs))

print("\nFirst 20 genes:")
print(genes[:20])

print("\nFirst 20 expressions:")
print(exprs[:20])

print("\nLast 20 genes:")
print(genes[-20:])

print("\nLast 20 expressions:")
print(exprs[-20:])


split: train
label: 20
target_gene: CACNA1C
drug: Berbamine
cell_line_id: CVCL_0099
sample: smp_1786
plate: plate4

Genes length: 713
Expressions length: 713

First 20 genes:
[ 20  21  59  62  82  95 108 149 151 182 221 246 250 252 262 273 294 298
 300 312]

First 20 expressions:
[0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 1.09861231 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 1.38629436 0.69314718]

Last 20 genes:
[47641 48361 50144 51002 51551 52326 53535 54325 54791 56819 57408 57644
 57680 57689 58015 58375 59038 59535 61490 62326]

Last 20 expressions:
[0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 1.09861231 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718]


In [4]:
gene_expr_df = pd.DataFrame({
    "gene_id": genes,
    "expression_log1p": exprs
})

# 按表达量排序（理论上应当是“高表达优先，但不保证完全有序”）
gene_expr_df_sorted = gene_expr_df.sort_values(
    "expression_log1p", ascending=False
)

gene_expr_df_sorted.head(20)

,gene_id,expression_log1p
1462,21401,4.779123
1464,21437,4.454347
1067,11268,2.833213
1521,39721,2.639057
205,1721,2.484907
240,2062,2.484907
170,1318,2.397895
187,1512,2.397895
163,1278,2.302585
1514,37295,2.197225
